# Can we explain it? Decision trees on real bookings

CL2 of *Predictive Analytics*. Work in **pairs**; swap at every station: **predict, run, change, decide.**

**Read** questions are in the style of the exam: answer them on paper first, then run the cell below.

**Stuck?** Every station only needs the *Setup* cell. Just go on with the next station.

| Station | Question | Time |
|---|---|---|
| 1 | The tree that knows everything | 8 min |
| 2 | How deep? | 14 min |
| 3 | Explain it | 13 min |
| 4 | Promise and test | 10 min |
| Homework | Is the tree stable? | 15 min |
| Extra | For the fast ones | |

> *From: Mara, hotel manager. Not too many questions, I understood that. By tonight I need the rules for the reception on one page and an honest promise per 1,000 bookings. And tell me: does it cost us money that the rules are so simple?*

Data and costs as in CL1: the offer costs **30 €**, a prevented cancellation is worth **100 €** (fictional), so we make the offer if p ≥ **0.3**. The bookings are split anew. The leak of CL1 (`parking_spaces`) is left out. Data: Antonio, Almeida, Nunes 2019, [Hotel booking demand datasets](https://doi.org/10.1016/j.dib.2018.11.126), CC BY 4.0, cleaned, a random sample.

## Setup

Run this cell once (about 10 seconds). You do not have to read the code.

In [ ]:
#@title Setup (run me) { display-mode: "form" }
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

URL = "https://fjoelsak.github.io/thm_wk-1609-predictiveanalytics/cl1-who-gets-the-offer/hotel-bookings.csv"
FILE = Path("hotel-bookings.csv")
df = pd.read_csv(FILE if FILE.exists() else URL)

# 60 % train, 20 % validation (for decisions), 20 % test (only for the last station)
train, rest = train_test_split(df, test_size=0.4, random_state=1, stratify=df["cancelled"])
valid, _test = train_test_split(rest, test_size=0.5, random_state=1, stratify=rest["cancelled"])

OFFER_COST, SAVED = 30, 100
CATEGORICAL = ["hotel", "month", "meal", "channel", "customer_type", "room_type"]
NUMERIC = [c for c in df.columns if c not in CATEGORICAL + ["cancelled", "parking_spaces"]]
ALL = NUMERIC + CATEGORICAL


def profit_per_1000(y, offered):
    """Profit per 1,000 bookings: +70 for each offer to a canceller, -30 for each other offer."""
    y, offered = np.asarray(y), np.asarray(offered, dtype=bool)
    return 1000 * np.mean(offered * (SAVED * y - OFFER_COST))


def report(y, offered):
    """Confusion matrix and measures for a list of offers."""
    tn, fp, fn, tp = confusion_matrix(y, offered, labels=[0, 1]).ravel()
    print(pd.DataFrame([[tn, fp], [fn, tp]], index=["came", "cancelled"], columns=["no offer", "offer"]), "\n")
    print(f"sensitivity {tp / (tp + fn):.2f}   specificity {tn / (tn + fp):.2f}   "
          f"precision {tp / (tp + fp) if tp + fp else float('nan'):.2f}   accuracy {(tp + tn) / len(y):.2f}")
    print(f"profit per 1,000 bookings: {profit_per_1000(y, offered):.0f} €")


def check(name, value, expected, tolerance=0.01):
    """Compare your value with the expected one."""
    ok = value is not None and abs(value - expected) <= tolerance
    print(f"{name}:", "correct ✅" if ok else "not yet ❌ (have another look at the output above)")


def hand_in(y, offered, decision, promise):
    """Last station: report on the test data and print the line to hand in."""
    report(y, offered)
    profit = profit_per_1000(y, offered)
    print(f"\nHAND IN >>> {decision} | promised {promise:.0f} € | reached {profit:.0f} € "
          f"| score {profit - abs(profit - promise):.0f}")


from sklearn.tree import DecisionTreeClassifier, plot_tree


def train_tree(features=ALL, data=train, **brakes):
    """Decision tree, e.g. train_tree(max_depth=3); categories are one-hot encoded, no scaling."""
    num = [f for f in features if f in NUMERIC]
    cat = [f for f in features if f in CATEGORICAL]
    prepare = make_column_transformer(("passthrough", num), (OneHotEncoder(drop="if_binary"), cat))
    return make_pipeline(prepare, DecisionTreeClassifier(random_state=0, **brakes)).fit(data[features], data["cancelled"])


def train_logreg():
    """Logistic regression on all features, as in CL1."""
    prepare = make_column_transformer((StandardScaler(), NUMERIC), (OneHotEncoder(drop="first"), CATEGORICAL))
    return make_pipeline(prepare, LogisticRegression(max_iter=2000)).fit(train[ALL], train["cancelled"])


def proba(model, data):
    """Predicted probability of a cancellation."""
    return model.predict_proba(data[model.feature_names_in_])[:, 1]


def accuracy(model, data):
    """Accuracy with the threshold 0.5."""
    return model.score(data[model.feature_names_in_], data["cancelled"])


def names_of(model):
    """Column names after the preparation, without the prefix of the transformer."""
    return [n.split("__")[1] for n in model[0].get_feature_names_out()]


def rules(model):
    """Print every leaf of a (small) tree as a rule."""
    t, names = model[-1].tree_, names_of(model)

    def walk(node, path):
        if t.children_left[node] < 0:
            share = t.value[node][0][1] / t.value[node][0].sum()
            print(f"if {' and '.join(path)}\n   -> p = {share:.2f}   ({t.n_node_samples[node]} bookings)")
            return
        name, cut = names[t.feature[node]], t.threshold[node]
        binary = name not in NUMERIC
        walk(t.children_left[node], path + [f"not {name}" if binary else f"{name} <= {cut:.1f}"])
        walk(t.children_right[node], path + [name if binary else f"{name} > {cut:.1f}"])

    walk(0, [])


def final_test(model, threshold, promise):
    """Station 4: evaluate the decision on the test data."""
    tree = model[-1]
    size = f"depth {tree.get_depth()}, {tree.get_n_leaves()} leaves" if hasattr(tree, "get_depth") else "logistic regression"
    hand_in(_test["cancelled"], proba(model, _test) >= threshold, f"{size}, threshold {threshold}", promise)


print(f"{len(train)} train, {len(valid)} validation, {len(_test)} test bookings")
df.head()

## Station 1: The tree that knows everything

We train a tree **without any brake**: it asks questions until every leaf is pure.

**Predict:** which accuracy does it reach on the training data, which on the validation data? (Offer to nobody: 73 %.)

**Prediction:** *write here*

In [ ]:
tree = train_tree()
print(f"accuracy on training data:   {accuracy(tree, train):.3f}")
print(f"accuracy on validation data: {accuracy(tree, valid):.3f}   (offer to nobody: {1 - valid['cancelled'].mean():.3f})")
print(f"depth {tree[-1].get_depth()}, {tree[-1].get_n_leaves()} leaves for {len(train)} bookings")

**Change:** put in one brake, for example `max_depth=3` or `min_samples_leaf=50`. What happens to the two accuracies and to the number of leaves?

In [ ]:
tree = train_tree(max_depth=10)  # change me

print(f"training {accuracy(tree, train):.3f}   validation {accuracy(tree, valid):.3f}   {tree[-1].get_n_leaves()} leaves")

**Read:** what does the code print?

```python
y    = np.array([1, 0, 0, 1, 0])
pred = np.array([1, 0, 1, 1, 1])
print((pred == y).mean())
```

**Answer:** *write here*

In [ ]:
y = np.array([1, 0, 0, 1, 0])
pred = np.array([1, 0, 1, 1, 1])
print((pred == y).mean())

**Decide:** explain to Mara in one sentence why the 'perfect' tree must not go live.

**Answer:** *write here*

## Station 2: How deep?

**Predict:** sketch two curves over the depth 1 to 20: accuracy on training and on validation data. Where is the best depth?

**Prediction:** *write here*

In [ ]:
depths = range(1, 21)
trees = [train_tree(max_depth=d) for d in depths]

plt.plot(depths, [accuracy(t, train) for t in trees], color="gray", linewidth=3, label="training")
plt.plot(depths, [accuracy(t, valid) for t in trees], color="green", linewidth=3, label="validation")
plt.axhline(1 - valid["cancelled"].mean(), linestyle="--", color="gray", label="offer to nobody")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.legend()
plt.show()

Accuracy is not what we earn. The same trees with the threshold 0.3:

In [ ]:
depths = range(1, 21)
trees = [train_tree(max_depth=d) for d in depths]

profits = [profit_per_1000(valid["cancelled"], proba(t, valid) >= 0.3) for t in trees]
plt.plot(depths, profits, color="green", linewidth=3, label="trees")
plt.axhline(profit_per_1000(valid["cancelled"], proba(train_logreg(), valid) >= 0.3), linestyle="--", color="black", label="logistic regression")
plt.xlabel("max_depth")
plt.ylabel("profit per 1,000 bookings in €")
plt.legend()
plt.show()

**Change:** combine two brakes, for example `max_depth` and `min_samples_leaf`. How close do you get to the logistic regression?

In [ ]:
tree = train_tree(max_depth=4, min_samples_leaf=1)  # change me

print(f"tree:                {profit_per_1000(valid['cancelled'], proba(tree, valid) >= 0.3):.0f} €   ({tree[-1].get_n_leaves()} leaves)")
print(f"logistic regression: {profit_per_1000(valid['cancelled'], proba(train_logreg(), valid) >= 0.3):.0f} €")

**Read:** what does the code print, and why is this the right way to choose the depth?

```python
depths = [1, 2, 3, 4, 5]
train_acc = np.array([0.75, 0.79, 0.82, 0.88, 0.95])
valid_acc = np.array([0.74, 0.78, 0.79, 0.77, 0.73])
print(depths[np.argmax(valid_acc)])
```

**Answer:** *write here*

In [ ]:
depths = [1, 2, 3, 4, 5]
valid_acc = np.array([0.74, 0.78, 0.79, 0.77, 0.73])
print(depths[np.argmax(valid_acc)])

**Decide:** which brakes do you take? Give one reason.

**Answer:** *write here*

## Station 3: Explain it

**Predict:** which feature will be the **root** of the tree?

**Prediction:** *write here*

In [ ]:
tree = train_tree(max_depth=3)
plt.figure(figsize=(17, 7))
plot_tree(tree[-1], feature_names=names_of(tree), class_names=["comes", "cancels"], filled=True, rounded=True, proportion=True, fontsize=8)
plt.show()

The same tree as rules, one per leaf:

In [ ]:
tree = train_tree(max_depth=3)
rules(tree)

**Change:** use the rules (not the computer!): with which probability is a booking cancelled that comes from a **portal**, **60 days** before arrival, as a single booking (`Transient`)? Enter your value.

In [ ]:
p_booking = None  # your result

new = valid.iloc[[0]].assign(channel="portal", lead_time=60, customer_type="Transient")
check("Probability", p_booking, proba(train_tree(max_depth=3), new)[0])

**Read:** which class is predicted for a booking with `portal = 1` and `lead_time = 10`?

```
|--- portal <= 0.50
|   |--- class: 0
|--- portal >  0.50
|   |--- lead_time <= 14.50
|   |   |--- class: 0
|   |--- lead_time >  14.50
|   |   |--- class: 1
```

**Answer:** *write here*

**Decide:** write **two** rules for the reception in plain words, one for making the offer and one for not making it. The tree almost always predicts `comes` (why?), but we use the threshold 0.3.

**Answer:** *write here*

## Station 4: Promise and test

Choose your tree, your threshold and **promise** Mara a profit per 1,000 bookings. `final_test` uses bookings that nobody has seen so far. **One shot.**

**Score = profit − |profit − promise|.**

In [ ]:
my_tree = train_tree(max_depth=2)  # your decision
my_threshold = 0.5                 # your decision
my_promise = 0                     # your promise in € per 1,000 bookings

# final_test(my_tree, my_threshold, my_promise)   # remove the first # when you are sure

**Hand in:** copy the line `HAND IN >>> ...` and add **one sentence**: tree or logistic regression, which one should go live?

**Answer:** *write here*

## Homework: Is the tree stable?

Run the *Setup* cell first. We train ten trees, each on a random sample (drawn with replacement) of the training bookings.

**Predict:** how many of the ten trees agree on whether a guest gets the offer? Think of a small tree (depth 3) and of a tree without brakes.

**Prediction:** *write here*

In [ ]:
def disagreement(**brakes):
    """Share of validation bookings for which ten trees do not all make the same decision."""
    offers = [proba(train_tree(data=train.sample(frac=1, replace=True, random_state=s), **brakes), valid) >= 0.3 for s in range(10)]
    votes = np.mean(offers, axis=0)
    return np.mean((votes > 0) & (votes < 1))


print(f"depth 3:        {disagreement(max_depth=3):.0%} of the guests are treated differently")
print(f"without brakes: {disagreement():.0%} of the guests are treated differently")

**Decide:** what does this mean for a guest? And what could we do with ten trees that contradict each other?

**Answer:** *write here*

## Extra: for the fast ones

1. **Entropy instead of Gini:** compare `train_tree(max_depth=5, criterion="entropy")` with the default. How large is the difference?
2. **Post-pruning:** try `train_tree(ccp_alpha=...)` with values between 0.0005 and 0.01.
3. **Exam question:** why does scaling `price` not change the tree?

In [ ]:
# your space